# 00. 데이터 정리 — 흡연과 간 수치(감마지티피)

**자료탐색방법론 기말 팀 프로젝트 · 8조**
주제: 흡연자의 간 수치가 높은 건 담배 탓일까, 술 탓일까?

이 노트북은 원자료를 불러와 **분석에 쓸 6개 변수만 골라 정리하고 `hc2023_clean.csv`로 저장**합니다. 질문 1~3 노트북은 모두 이 파일에서 시작합니다.

## 자료 정보
| 항목 | 내용 |
|---|---|
| 자료명 | 국민건강보험공단_건강검진정보 (2023년) |
| 제공기관 | 국민건강보험공단 |
| 원출처 | 공공데이터포털 https://www.data.go.kr/data/15007122/fileData.do (2026-11-17 기준 페이지 열리지 않음) |
| 실제 다운로드 경로 | https://www.gimi9.com/dataset/www-data-go-kr-data-filedata-15007122/ (공공데이터포털 사본) |
| 다운로드 날짜 | 2026-11-17 |
| 파일 | `국민건강보험공단_건강검진정보_2023.zip` (CSV 1개 + 사용자 매뉴얼 hwpx 1개), GitHub에는 `hc2023.zip`으로 이름만 바꿔 올림 |
| 이용 조건 | 이용허락범위 제한 없음 |
| 수집 맥락 | 2023년 일반건강검진 수검자 중 **100만 명 무작위 추출**, 한 시점 자료(단면자료) |
| **한 행의 의미** | **2023년 건강검진을 받은 사람 1명** |

## 실행 순서
1. 아래 첫 코드 셀이 GitHub 저장소(https://github.com/wjaioajqdo12-coder/eda-team8-smoking)의 Releases(`data`)에서 zip 파일을 자동으로 내려받습니다.
2. 내려받기가 안 되면 `RAW_URL = ""`로 비우고 실행하세요. 파일 업로드 창이 뜨면 zip 파일을 직접 올리면 됩니다.
3. 위에서부터 순서대로 끝까지 실행합니다. (런타임 → 모두 실행)

In [ ]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd

ZIP_NAME = "hc2023.zip"   # 원본 이름: 국민건강보험공단_건강검진정보_2023.zip (주소에 한글이 있으면 내려받기가 실패해서 영어 이름으로 바꿔 올림)
RAW_URL = "https://github.com/wjaioajqdo12-coder/eda-team8-smoking/releases/download/data/hc2023.zip"   # 웹 업로드 한도(25MB)를 넘어서 Releases에 올림

if not os.path.exists(ZIP_NAME):
    if RAW_URL:
        urllib.request.urlretrieve(RAW_URL, ZIP_NAME)
        assert zipfile.is_zipfile(ZIP_NAME), "zip이 아닌 파일이 받아졌습니다. RAW_URL을 확인하세요"
    else:
        try:
            from google.colab import files
            uploaded = files.upload()          # zip 파일 선택
            ZIP_NAME = list(uploaded.keys())[0]
        except ImportError:
            raise FileNotFoundError("zip 파일을 노트북과 같은 폴더에 두거나 RAW_URL을 입력하세요")

print("사용할 파일:", ZIP_NAME, f"({os.path.getsize(ZIP_NAME)/1e6:.1f} MB)")

## 1. 원자료 불러오기
zip 안의 파일 이름이 한글이라 압축을 풀면 깨질 수 있습니다. 그래서 풀지 않고 zip 안에서 바로 CSV를 읽습니다. 인코딩은 `cp949`입니다.

In [ ]:
with zipfile.ZipFile(ZIP_NAME) as zf:
    csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
    assert len(csv_members) == 1, f"CSV가 1개가 아닙니다: {csv_members}"
    with zf.open(csv_members[0]) as f:
        raw = pd.read_csv(f, encoding="cp949")

print("원자료 크기(행, 열):", raw.shape)
print("기준년도:", raw["기준년도"].unique())
print("가입자일련번호 중복 없음:", raw["가입자일련번호"].is_unique)
raw.head()

## 2. 자료 사전 (분석에 쓰는 6개 변수)
코드 설명은 zip 안의 「건강검진정보 사용자 매뉴얼」에서 가져왔습니다.

| 새 이름 | 원래 열 | 의미 | 값과 단위 | 척도 |
|---|---|---|---|---|
| `성별` | 성별코드 | 성별 | 1 남자, 2 여자 | 명목 |
| `연령대` | 연령대코드(5세단위) | 나이 5세 구간 | 5 = 20~24세 … 17 = 80~84세, 18 = 85세 이상 | 순서 |
| `시도` | 시도코드 | 거주 시도 | 11 서울 … 49 제주 (17개) | 명목 |
| `감마지티피` | 감마지티피 | 간 기능 혈액검사 수치 | IU/L, 정상치 남 11~63 / 여 8~35 | 비율 |
| `흡연` | 흡연상태 | 문진 응답 | 1 비흡연, 2 과거 흡연(끊음), 3 현재 흡연 | 명목(순서로도 볼 수 있음) |
| `음주` | 음주여부 | 문진 응답 | 0 마시지 않음, 1 마심 | 명목 |

**주의:** 흡연과 음주는 본인이 답한 값입니다. 음주는 마시는 양이 아니라 여부만 있습니다.

In [ ]:
rename = {
    "성별코드": "성별코드", "연령대코드(5세단위)": "연령대코드", "시도코드": "시도코드",
    "감마지티피": "감마지티피", "흡연상태": "흡연코드", "음주여부": "음주코드",
}
df = raw[list(rename)].rename(columns=rename).copy()
df.insert(0, "id", raw["가입자일련번호"])
print(df.shape)
df.head()

## 3. 코드값 점검
매뉴얼에 없는 코드가 섞여 있지 않은지 확인합니다. `dropna=False`로 결측(NaN)도 함께 셉니다.

In [ ]:
expected = {
    "성별코드": {1, 2},
    "연령대코드": set(range(5, 19)),
    "시도코드": {11, 26, 27, 28, 29, 30, 31, 36, 41, 42, 43, 44, 45, 46, 47, 48, 49},
    "흡연코드": {1, 2, 3},
    "음주코드": {0, 1},
}
for col, ok in expected.items():
    vals = set(df[col].dropna().unique())
    unexpected = vals - ok
    print(f"{col}: 결측 {df[col].isna().sum():,}개, 매뉴얼에 없는 코드 {sorted(unexpected) if unexpected else '없음'}")
    assert not unexpected

## 4. 결측 점검 (전체 수 vs 유효 수)
2차수에서 배운 대로 `size`(전체)와 `count`(결측 제외)를 구분해서 적습니다.

In [ ]:
missing = pd.DataFrame({
    "전체(size)": len(df),
    "유효(count)": df.count(),
    "결측": df.isna().sum(),
})
missing["결측 비율(%)"] = (missing["결측"] / missing["전체(size)"] * 100).round(3)
missing

## 5. 감마지티피 극단값 점검: 9999는 결측 표시일까?
최댓값이 정확히 9999인 사람이 있습니다. 매뉴얼에는 9999에 대한 설명이 없습니다.
- **예상:** 입력 안 된 값을 9999로 채운 결측 표시일 것이다.
- **확인 절차:** 9999 바로 아래 값들이 어떻게 이어지는지 보고, 9999인 사람의 다른 간 수치(AST, ALT)를 확인한다. 결측 표시라면 AST·ALT는 평범할 것이다.

In [ ]:
g = df["감마지티피"]
print("1000 이상:", (g >= 1000).sum(), "명 / 9999:", (g == 9999).sum(), "명")
print("\n9999를 뺀 가장 큰 값 5개:", sorted(g[g < 9999].dropna())[-5:])

check = raw.loc[g == 9999, ["성별코드", "연령대코드(5세단위)", "혈청지오티(AST)", "혈청지피티(ALT)", "흡연상태", "음주여부"]]
print("\nAST·ALT 정상치는 0~40 IU/L")
print("AST 또는 ALT가 40 초과인 사람:", ((check["혈청지오티(AST)"] > 40) | (check["혈청지피티(ALT)"] > 40)).sum(), "/", len(check), "명")
check

**판단:** 9999인 13명 모두 AST 또는 ALT가 정상치(40)보다 높습니다. 또 9999 바로 아래 값은 3300이라 그 사이가 비어 있습니다. 그래서 결측 표시라기보다 **실제로 매우 높은 값을 입력 상한(9999)으로 적은 것**으로 보는 게 더 그럴듯합니다.

**처리 기준:**
- 삭제하지 않고 그대로 둡니다. (1차수: 큰 값 ≠ 자동 오류 ≠ 자동 삭제)
- `감마지티피_상한` 열에 표시해 두고, 평균처럼 극단값에 민감한 요약을 쓸 때는 이 13명을 뺀 결과도 함께 확인합니다.
- 중앙값·사분위수는 100만 명 중 13명에 거의 영향받지 않습니다.

이 과정은 검증 로그 후보입니다(예상: 결측 표시 → 실제: 상한 표기로 판단 → 수정: 결측 처리하지 않고 표시만 함).

In [ ]:
df["감마지티피_상한"] = df["감마지티피"].eq(9999)
print("표시된 사람:", df["감마지티피_상한"].sum())

## 6. 코드를 글자로 바꾸기
그래프와 표에서 읽기 쉽게 글자 열을 추가합니다. 원래 코드 열도 그대로 남겨서 언제든 대조할 수 있게 합니다. 연령대·흡연은 순서가 있는 범주로 만들어 표와 그림에서 순서가 섞이지 않게 합니다.

In [ ]:
sex_map = {1: "남", 2: "여"}
age_map = {c: f"{(c-1)*5}~{(c-1)*5+4}세" for c in range(5, 18)}
age_map[18] = "85세 이상"
sido_map = {11: "서울", 26: "부산", 27: "대구", 28: "인천", 29: "광주", 30: "대전", 31: "울산", 36: "세종",
            41: "경기", 42: "강원", 43: "충북", 44: "충남", 45: "전북", 46: "전남", 47: "경북", 48: "경남", 49: "제주"}
smk_map = {1: "비흡연", 2: "과거 흡연", 3: "현재 흡연"}
drk_map = {0: "비음주", 1: "음주"}

df["성별"] = df["성별코드"].map(sex_map)
df["연령대"] = pd.Categorical(df["연령대코드"].map(age_map), categories=[age_map[c] for c in range(5, 19)], ordered=True)
df["시도"] = df["시도코드"].map(sido_map)
df["흡연"] = pd.Categorical(df["흡연코드"].map(smk_map), categories=list(smk_map.values()), ordered=True)
df["음주"] = pd.Categorical(df["음주코드"].map(drk_map), categories=list(drk_map.values()))

# 코드가 있는데 글자가 비면 매핑 실수
for code_col, label_col in [("성별코드","성별"),("연령대코드","연령대"),("시도코드","시도"),("흡연코드","흡연"),("음주코드","음주")]:
    assert (df[code_col].notna() == df[label_col].notna()).all(), label_col
print("매핑 확인 완료")
df.head()

## 7. 손으로 대조하기 (12명)
무작위로 12명을 뽑아 원자료의 코드와 바꾼 글자를 나란히 놓습니다. 매뉴얼 표를 보면서 한 줄씩 맞는지 직접 확인합니다. `random_state=8`로 고정해서 누가 실행해도 같은 12명이 나옵니다.

In [ ]:
sample12 = df.sample(12, random_state=8)[["id","성별코드","성별","연령대코드","연령대","시도코드","시도","흡연코드","흡연","음주코드","음주","감마지티피"]]
sample12

## 8. 로그 변환 준비
감마지티피는 오른쪽으로 길게 늘어진 분포입니다(중앙값 약 22, 최댓값 9999). 로그는 0 이하 값에 쓸 수 없으니 먼저 확인합니다.

In [ ]:
print("0 이하 값:", (df["감마지티피"] <= 0).sum(), "개")
print("최솟값:", df["감마지티피"].min())
df["log10_감마지티피"] = np.log10(df["감마지티피"])
df[["감마지티피", "log10_감마지티피"]].describe().round(3)

0 이하 값이 없어서 `log10(감마지티피)`를 그대로 씁니다. 상용로그를 쓰면 1 = 10, 2 = 100 IU/L로 읽을 수 있어 해석이 쉽습니다. 다른 변환(제곱근 등)과의 비교는 질문 1 노트북에서 합니다.

## 9. 결측 처리 원칙
`df.dropna()`로 한꺼번에 지우면 질문과 상관없는 열 때문에 사람이 빠집니다(2차수 Module 3). 그래서 **정리 파일에는 결측을 그대로 두고, 질문마다 필요한 열에서만 뺍니다.** 아래 함수를 각 질문 노트북에서 쓰면 됩니다.

In [ ]:
def for_question(data, cols):
    """질문에 필요한 열에서만 결측을 빼고, 몇 명이 빠졌는지 출력"""
    out = data.dropna(subset=cols)
    print(f"전체 {len(data):,}명 → 분석 {len(out):,}명 (제외 {len(data)-len(out):,}명, 기준 열: {cols})")
    return out

q1 = for_question(df, ["감마지티피"])
q2 = for_question(df, ["감마지티피", "흡연", "성별", "음주"])
q3 = for_question(df, ["흡연", "시도", "성별", "연령대"])

## 10. 층별 인원 확인 (질문 2, 3 준비)
층화할 때 집단마다 몇 명인지 먼저 봅니다. 인원이 아주 적은 칸은 결과가 불안정하니 해석할 때 주의합니다.

In [ ]:
cells_q2 = q2.groupby(["성별", "음주", "흡연"], observed=True).size().unstack("흡연")
print("질문 2: 성별 × 음주 × 흡연 인원")
cells_q2

In [ ]:
cells_q3 = q3.groupby("시도").size().sort_values(ascending=False).rename("인원")
print("질문 3: 시도별 인원 (흡연율의 분모)")
cells_q3.to_frame().T

## 11. 정리 파일 저장

In [ ]:
keep = ["id", "성별코드", "성별", "연령대코드", "연령대", "시도코드", "시도",
        "감마지티피", "log10_감마지티피", "감마지티피_상한", "흡연코드", "흡연", "음주코드", "음주"]
clean = df[keep]
clean.to_csv("hc2023_clean.csv", index=False, encoding="utf-8-sig")
print("저장:", clean.shape, f"{os.path.getsize('hc2023_clean.csv')/1e6:.1f} MB")

# 다시 읽어서 원자료와 행 수·값이 같은지 확인
back = pd.read_csv("hc2023_clean.csv", encoding="utf-8-sig")
assert len(back) == len(raw)
assert (back["감마지티피"].fillna(-1).values == raw["감마지티피"].fillna(-1).values).all()
print("저장 파일 확인 완료: 행 수와 감마지티피 값이 원자료와 같음")

## 정리 기준 요약 (보고서·발표에 그대로 옮겨 쓰기)
1. 분석 단위: 2023년 건강검진 수검자 1명 = 1행, 총 1,000,000명
2. 사용 변수: 성별, 연령대, 시도, 감마지티피, 흡연상태, 음주여부
3. 결측: 정리 파일에는 그대로 두고, 질문별로 필요한 열에서만 제외 (제외 인원은 각 질문 노트북에 기록)
4. 감마지티피 9999(13명): 실제 높은 값의 상한 표기로 판단해 유지, `감마지티피_상한` 열로 표시
5. 로그 변환: 0 이하 값 없음 → `log10(감마지티피)` 사용
6. 해석 한계: 한 시점 자료(단면)이고 흡연·음주는 자기 응답이라 원인까지는 말할 수 없음. 음주는 양이 아닌 여부만 있음

## 질문 노트북에서 시작하는 법
`hc2023_clean.csv`는 약 92MB라 GitHub에는 올리지 않고, 각자 이 노트북을 먼저 실행해서 만듭니다. GitHub에는 원본 zip(약 36MB)과 노트북만 올립니다.

```python
import pandas as pd
df = pd.read_csv("hc2023_clean.csv", encoding="utf-8-sig")
```